# Comparison of perm2code Implementations

Comparision between `perm2code` and `perm2code_2`.

In [1]:
import time

import numpy as np

from lehmer import Lehmer

In [2]:
def compare_functions(
    lc: Lehmer,
    test_vector,
    name1: str = "Function 1",
    name2: str = "Function 2",
    number: int = 100,
    repeat: int = 3,
) -> dict:

    # Warmup
    lc.perm2code(test_vector)
    lc.perm2code_2(test_vector)

    times1 = []
    for _ in range(repeat):
        start = time.perf_counter()
        for _ in range(number):
            lc.perm2code(test_vector)
        end = time.perf_counter()
        times1.append((end - start) / number)

    times2 = []
    for _ in range(repeat):
        start = time.perf_counter()
        for _ in range(number):
            lc.perm2code_2(test_vector)
        end = time.perf_counter()
        times2.append((end - start) / number)

    mean1 = np.mean(times1)
    mean2 = np.mean(times2)
    return mean1.item(), mean2.item()


In [3]:
def compare(n, batch, number=100, repeat=20):
    lc = Lehmer(n=n)
    test_vector = [np.random.permutation(n) for _ in range(batch)]
    perm2code, perm2code_2 = compare_functions(lc, test_vector, number=number, repeat=repeat)
    return perm2code, perm2code_2

In [4]:
results = {}
# Just for the sake of the comparision. Some combinations might not make sense.
batches = [1, 5, 10, 20, 50, 100, 500]
Ns = [3, 4, 5, 8, 10, 15, 20, 30, 50, 100, 200, 500]
for n in Ns:
    for batch in batches:
        perm2code_time, perm2code_2_time = compare(n, batch)
        results[(n, batch)] = (perm2code_time, perm2code_2_time)

In [5]:
import pandas as pd

table_data = []
for n in Ns:
    row = {'n': n}
    for batch in batches:
        perm2code_time, perm2code_2_time = results[(n, batch)]
        row[f'b={batch}'] = f'{perm2code_time*1e6/batch:.2f} / {perm2code_2_time*1e6/batch:.2f}'
    table_data.append(row)

df = pd.DataFrame(table_data)

In [6]:
def color_cell(val):
    if '/' not in str(val):
        return ''
    parts = str(val).split(' / ')
    time1 = float(parts[0])
    time2 = float(parts[1])
    if time1 < time2:
        return 'color: green'
    else:
        return 'color: red'


The following table displays the timing results for the two implementations of the `perm2code` function. The results, average time in microseconds,  are shown side by side like this `perm2code/perm2code_2`. They are normalized to the batch side. Green cells indicate that `perm2code` was faster. `n` is the length of the permutation and `b` is the batch size (number of permutations processed simultaneously).

![](./figures/table.png)

In [7]:

df.style.applymap(color_cell, subset=[col for col in df.columns if col != 'n'])

/var/folders/ml/lm19ddxj2tg9_zt3rj4z3m_00000gn/T/ipykernel_69419/201831833.py:1: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  df.style.applymap(color_cell, subset=[col for col in df.columns if col != 'n'])


,n,b=1,b=5,b=10,b=20,b=50,b=100,b=500
0,3,11.51 / 8.27,1.99 / 2.00,1.14 / 1.02,0.69 / 0.59,0.37 / 0.31,0.29 / 0.24,0.22 / 0.16
1,4,9.55 / 12.13,2.02 / 2.49,1.15 / 1.30,0.78 / 0.77,0.42 / 0.41,0.31 / 0.27,0.24 / 0.18
2,5,9.88 / 13.97,2.09 / 3.04,1.21 / 1.60,0.70 / 0.90,0.43 / 0.47,0.34 / 0.32,0.27 / 0.21
3,8,9.96 / 22.20,2.40 / 4.92,1.30 / 2.59,0.79 / 1.46,0.53 / 0.71,0.44 / 0.50,0.33 / 0.38
4,10,9.78 / 37.53,2.33 / 6.19,1.69 / 3.18,0.85 / 1.79,0.63 / 0.91,0.52 / 0.57,0.39 / 0.35
5,15,10.25 / 42.99,2.62 / 9.14,2.05 / 5.14,1.30 / 2.64,0.91 / 1.26,0.79 / 0.82,1.05 / 0.60
6,20,10.92 / 55.51,2.91 / 12.02,1.90 / 6.41,1.55 / 3.46,1.14 / 1.78,0.99 / 1.08,0.88 / 0.70
7,30,11.98 / 83.44,4.45 / 18.23,3.05 / 9.43,2.42 / 5.05,1.98 / 2.54,1.83 / 2.37,1.77 / 1.15
8,50,14.96 / 190.44,13.89 / 34.21,5.96 / 16.50,4.91 / 9.42,4.41 / 4.74,4.18 / 3.50,4.21 / 2.53
9,100,33.34 / 306.09,19.79 / 68.44,16.57 / 37.65,16.05 / 21.08,14.69 / 12.47,13.90 / 10.39,14.66 / 6.97
